# RNN e LSTM  
 

 

In [ ]:
import math
import re
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(0)
np.random.seed(0)


## 1) Dataset testo (character-level) + Vocab
Per evitare dipendenze e download, usiamo un piccolo testo integrato nel notebook.

### Tokenizzazione
Qui facciamo **char-level**: ogni carattere è un token.

### Vocab
La `Vocab` mappa:
- char → id
- id → char

In [ ]:

TEXT = r"""
The Time Traveller (for so it will be convenient to speak of him) was expounding a recondite matter to us.
His grey eyes shone and twinkled, and his usually pale face was flushed and animated.
The Fire burned brightly, and the soft radiance of the incandescent lights in the lilies of silver caught the bubbles
that flashed and passed in our glasses. Our chairs, being his patents, embraced and caressed us rather than submitted
to be sat upon, and there was that luxurious after-dinner atmosphere when thought roams gracefully free of the trammels
of precision. And he put it to us in this way—marking the points with a lean forefinger—as we sat and lazily admired his earnestness
over this new paradox (as we thought it) and his fecundity.
"""

def clean_text(s: str) -> str:
    # Manteniamo lettere, punteggiatura di base e spazi; riduciamo spazi multipli
    s = s.replace("—", "-")
    s = re.sub(r"\s+", " ", s.strip()) #normalizza gli spazi in una stringa.
    return s

text = clean_text(TEXT)
print("Lunghezza testo (char):", len(text))
print(text[:200])


In [ ]:

class Vocab:
    def __init__(self, tokens):
        # tokens: lista di token (qui: caratteri)
        uniq = sorted(set(tokens))
        self.token_to_idx = {t:i for i,t in enumerate(uniq)}
        self.idx_to_token = uniq

    def __len__(self):
        return len(self.idx_to_token)

    def to_ids(self, tokens):
        return [self.token_to_idx[t] for t in tokens]

    def to_tokens(self, ids):
        return [self.idx_to_token[i] for i in ids]

tokens = list(text)
vocab = Vocab(tokens)

corpus = torch.tensor(vocab.to_ids(tokens), dtype=torch.long)
print("Vocab size:", len(vocab))
print("Primi 20 id:", corpus[:20].tolist())
print("Primi 20 token:", vocab.to_tokens(corpus[:20].tolist()))

## 2) Batch iterator per Language Modeling
Creiamo mini-batch di sequenze per predire il prossimo carattere.

Dato un corpus di id:
- input  `X`: sequenze di lunghezza `num_steps`
- target `Y`: gli stessi id shiftati di 1 (next char)

Qui usiamo un campionamento tipo *random sampling* (semplice e didattico).

In [ ]:

def get_lm_batches_random(corpus, batch_size=32, num_steps=35):
    # corpus: 1D tensor di token id
    # Creiamo tanti subsequences possibili, poi campioniamo batch casuali
    num_subseq = (len(corpus) - 1) // num_steps
    initial_indices = torch.arange(num_subseq) * num_steps
    initial_indices = initial_indices[torch.randperm(len(initial_indices))]

    def data(pos):
        return corpus[pos:pos+num_steps]

    for i in range(0, len(initial_indices), batch_size):
        batch_idx = initial_indices[i:i+batch_size]
        X = torch.stack([data(int(j)) for j in batch_idx])
        Y = torch.stack([data(int(j)+1) for j in batch_idx])
        yield X, Y

# test
X, Y = next(get_lm_batches_random(corpus, batch_size=4, num_steps=10))
print("X shape:", X.shape, "Y shape:", Y.shape)
print("X[0]:", "".join(vocab.to_tokens(X[0].tolist())))
print("Y[0]:", "".join(vocab.to_tokens(Y[0].tolist())))


## 3) Utility per LM: one-hot, grad clipping, predict
Per tenere il codice vicino a quello “classico” di RNN scratch, useremo input one-hot.

- `one_hot(X)`: `(B,T)` → `(T,B,V)`
- `grad_clip_`: evita esplosione dei gradienti
- `predict`: genera testo dato un prefisso

In [ ]:

def one_hot(X, vocab_size):
    # X: (B,T) long
    # output: (T,B,V) float
    return F.one_hot(X, num_classes=vocab_size).float().permute(1,0,2)

def grad_clip_(model, theta=1.0):
    params = [p for p in model.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum(p.grad**2) for p in params if p.grad is not None))
    if norm > theta:
        for p in params:
            if p.grad is not None:
                p.grad[:] *= theta / (norm + 1e-6)

@torch.no_grad()
def predict_lm(model, prefix, num_preds, vocab, device):
    model.eval()
    state = None
    output_ids = vocab.to_ids(list(prefix))
    # warm-up con prefix (eccetto ultimo char, che usiamo come input)
    for ch in output_ids[:-1]:
        X = torch.tensor([[ch]], device=device)
        y_hat, state = model(X, state)
    last = output_ids[-1]
    for _ in range(num_preds):
        X = torch.tensor([[last]], device=device)
        y_hat, state = model(X, state)  # y_hat: (B*T, V) o (B,T,V) dipende dal wrapper
        # y_hat da wrapper: (B, T, V) -> prendiamo ultimo timestep
        p = y_hat[0, -1].softmax(dim=-1)
        last = int(torch.multinomial(p, 1))
        output_ids.append(last)
    return "".join(vocab.to_tokens(output_ids))


## 4) RNN (by scratch)
Implementiamo:
\[ h_t = \tanh(X_t W_{xh} + h_{t-1} W_{hh} + b_h ) \]

Il wrapper `RNNLanguageModel`:
- prende input id `(B,T)`
- fa one-hot
- esegue la ricorrenza
- produce logits `(B,T,V)`, dove:
    - B = numero di sequenze nel batch
    - T = lunghezza della sequenza
    - V = dimensione vocabolario

In [ ]:

class RNNScratch(nn.Module):
    def __init__(self, vocab_size, hidden_size, sigma=0.01):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.W_xh = nn.Parameter(torch.randn(vocab_size, hidden_size) * sigma)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * sigma)
        self.b_h  = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, X_TBV, state=None):
        # X_TBV: (T,B,V)
        T, B, V = X_TBV.shape
        if state is None:
            H = torch.zeros(B, self.hidden_size, device=X_TBV.device)
        else:
            H = state
        outputs = []
        for t in range(T):
            X_t = X_TBV[t]  # (B,V)
            H = torch.tanh(X_t @ self.W_xh + H @ self.W_hh + self.b_h)
            outputs.append(H)
        # outputs: lista di (B,H) -> (T,B,H)
        return torch.stack(outputs, dim=0), H

class RNNLanguageModelScratch(nn.Module):
    def __init__(self, rnn_core: RNNScratch, vocab_size):
        super().__init__()
        self.rnn = rnn_core
        self.vocab_size = vocab_size
        self.hidden_size = rnn_core.hidden_size
        self.output = nn.Linear(self.hidden_size, vocab_size)

    def forward(self, X_BT, state=None):
        # X_BT: (B,T) token ids
        X_TBV = one_hot(X_BT, self.vocab_size).to(X_BT.device)  # (T,B,V)
        H_TBH, state = self.rnn(X_TBV, state)                   # (T,B,H)
        Y_TBV = self.output(H_TBH)                              # (T,B,V)
        return Y_TBV.permute(1,0,2), state                       # (B,T,V)

vocab_size = len(vocab)
rnn_core = RNNScratch(vocab_size=vocab_size, hidden_size=128).to(device)
lm_scratch = RNNLanguageModelScratch(rnn_core, vocab_size=vocab_size).to(device)

print(lm_scratch)
print(predict_lm(lm_scratch, "The ", 50, vocab, device))


### Training loop per Language Modeling
Useremo `CrossEntropyLoss` sui logits per-pixel (qui per carattere).

Nota su shape:
- logits: `(B,T,V)`
- target: `(B,T)`
Per `CrossEntropyLoss` conviene appiattire in:
- logits: `(B*T, V)`
- target: `(B*T,)`

In [ ]:

def train_lm(model, corpus, vocab, *, lr=1.0, num_epochs=30, batch_size=32, num_steps=35, clip=1.0):
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(1, num_epochs+1):
        total_loss, n_tokens = 0.0, 0
        for X, Y in get_lm_batches_random(corpus, batch_size=batch_size, num_steps=num_steps):
            X = X.to(device)
            Y = Y.to(device)

            optimizer.zero_grad()
            logits, _ = model(X)                # (B,T,V)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
            loss.backward()
            grad_clip_(model, theta=clip)
            optimizer.step()

            total_loss += loss.item() * Y.numel()
            n_tokens += Y.numel()

        ppl = math.exp(total_loss / n_tokens)
        if epoch in [1, 5, 10, num_epochs]:
            print(f"Epoch {epoch:3d} | ppl {ppl:.2f}")
            print("  sample:", predict_lm(model, "The ", 80, vocab, device))

# Esegui un training breve (aumenta epochs per più qualità)
train_lm(lm_scratch, corpus, vocab, lr=1.0, num_epochs=10, batch_size=64, num_steps=35, clip=1.0)


## 5) RNN con `nn.RNN`
Qui usiamo l'API high-level, ma manteniamo lo **stesso dataset** e lo **stesso training loop**.

Nota: `nn.RNN` si aspetta input `(T,B,D)` se `batch_first=False` (default). Qui teniamo `(B,T,V)` e impostiamo `batch_first=True`.

In [ ]:

class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.rnn = nn.RNN(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, X_BT, state=None):
        X_BTV = F.one_hot(X_BT, num_classes=self.vocab_size).float()
        out, state = self.rnn(X_BTV, state)      # out: (B,T,H)
        logits = self.output(out)                # (B,T,V)
        return logits, state

lm_rnn = RNNLanguageModel(vocab_size=len(vocab), hidden_size=128).to(device)
train_lm(lm_rnn, corpus, vocab, lr=1.0, num_epochs=10, batch_size=64, num_steps=35, clip=1.0)


## 6) LSTM (by scratch)
Implementiamo le gate principali:
- input gate, forget gate, output gate
- cell candidate

Equazioni (per timestep):
\[
I = \sigma(XW_{xi} + HW_{hi} + b_i),\;
F = \sigma(XW_{xf} + HW_{hf} + b_f),\;
O = \sigma(XW_{xo} + HW_{ho} + b_o),\;
\tilde{C} = \tanh(XW_{xc} + HW_{hc} + b_c)
\]
\[
C = F \odot C + I \odot \tilde{C},\;
H = O \odot \tanh(C)
\]

In [ ]:

class LSTMScratch(nn.Module):
    def __init__(self, vocab_size, hidden_size, sigma=0.01):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        def init_weight(*shape):
            return nn.Parameter(torch.randn(*shape) * sigma)
        def triple():
            return (init_weight(vocab_size, hidden_size),
                    init_weight(hidden_size, hidden_size),
                    nn.Parameter(torch.zeros(hidden_size)))

        self.W_xi, self.W_hi, self.b_i = triple()
        self.W_xf, self.W_hf, self.b_f = triple()
        self.W_xo, self.W_ho, self.b_o = triple()
        self.W_xc, self.W_hc, self.b_c = triple()

    def forward(self, X_TBV, state=None):
        T, B, V = X_TBV.shape
        if state is None:
            H = torch.zeros(B, self.hidden_size, device=X_TBV.device)
            C = torch.zeros(B, self.hidden_size, device=X_TBV.device)
        else:
            H, C = state

        outputs = []
        for t in range(T):
            X_t = X_TBV[t]

            I = torch.sigmoid(X_t @ self.W_xi + H @ self.W_hi + self.b_i)
            Fg = torch.sigmoid(X_t @ self.W_xf + H @ self.W_hf + self.b_f)
            O = torch.sigmoid(X_t @ self.W_xo + H @ self.W_ho + self.b_o)
            C_tilde = torch.tanh(X_t @ self.W_xc + H @ self.W_hc + self.b_c)

            C = Fg * C + I * C_tilde
            H = O * torch.tanh(C)

            outputs.append(H)

        return torch.stack(outputs, dim=0), (H, C)

class LSTMLanguageModelScratch(nn.Module):
    def __init__(self, lstm_core: LSTMScratch, vocab_size):
        super().__init__()
        self.lstm = lstm_core
        self.vocab_size = vocab_size
        self.output = nn.Linear(lstm_core.hidden_size, vocab_size)

    def forward(self, X_BT, state=None):
        X_TBV = one_hot(X_BT, self.vocab_size).to(X_BT.device)
        H_TBH, state = self.lstm(X_TBV, state)
        Y_TBV = self.output(H_TBH)
        return Y_TBV.permute(1,0,2), state

lstm_core = LSTMScratch(vocab_size=len(vocab), hidden_size=128).to(device)
lm_lstm_scratch = LSTMLanguageModelScratch(lstm_core, vocab_size=len(vocab)).to(device)

train_lm(lm_lstm_scratch, corpus, vocab, lr=1.0, num_epochs=10, batch_size=64, num_steps=35, clip=1.0)


---
# 7) Applicazione di Computer Vision: MNIST come sequenza (RNN/LSTM)
Ora applichiamo l’idea “sequenziale” alla CV.

- un'immagine MNIST 28×28 viene letta come **sequenza di 28 righe**
- ogni riga (28 pixel) è un timestep

Useremo una versione compatta con `nn.RNN` o `nn.LSTM`. 

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

def get_mnist():
    train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    print("MNIST trovato localmente (download=False).")
    return train_ds, test_ds
     
train_cv, test_cv = get_mnist()
train_loader = DataLoader(train_cv, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_cv, batch_size=128, shuffle=False)

# Visualizza esempi
xb, yb = next(iter(train_loader))
plt.figure(figsize=(4,4))
for i in range(4):
    plt.subplot(2,2,i+1)
    plt.imshow(xb[i,0].numpy(), cmap="gray")
    plt.title(f"y={yb[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Modello sequenziale per MNIST
Input: `(B, 1, 28, 28)` → `(B, 28, 28)`

Scegli `MODEL_CV = 'rnn'` oppure `'lstm'`.

In [ ]:
MODEL_CV = "lstm"  # "rnn" or "lstm"
hidden_cv = 128

class SeqMNIST(nn.Module):
    def __init__(self, cell="rnn", hidden_size=128, num_classes=10):
        super().__init__()
        self.cell = cell
        if cell == "rnn":
            self.rnn = nn.RNN(input_size=28, hidden_size=hidden_size, batch_first=True)
        elif cell == "lstm":
            self.rnn = nn.LSTM(input_size=28, hidden_size=hidden_size, batch_first=True)
        else:
            raise ValueError("cell must be 'rnn' or 'lstm'")
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, images):
        x = images.squeeze(1)         # (B,28,28) = (B,T,D)
        out, _ = self.rnn(x)
        # prendiamo l'ultimo timestep
        last = out[:, -1, :]          # (B,H)
        return self.fc(last)

model_cv = SeqMNIST(cell=MODEL_CV, hidden_size=hidden_cv).to(device)
opt = torch.optim.Adam(model_cv.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def eval_cv(model, loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return total_loss/total, correct/total

def train_cv(model, train_loader, test_loader, epochs=3):
    for ep in range(1, epochs+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        te_loss, te_acc = eval_cv(model, test_loader)
        print(f"[CV {MODEL_CV}] epoch {ep}/{epochs} | test loss {te_loss:.4f} acc {te_acc:.4f}")

train_cv(model_cv, train_loader, test_loader, epochs=3)


## Visualizzazione predizioni (MNIST sequenziale)

In [ ]:
model_cv.eval()
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)
with torch.no_grad():
    preds = model_cv(xb).argmax(1)

plt.figure(figsize=(7,7))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(xb[i,0].cpu().numpy(), cmap="gray")
    plt.title(f"vero={yb[i].item()} pred={preds[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()
